# 02 - Preprocess

Notebook này load dữ liệu, chia train/test và build preprocessing theo logic thật trong `ML_Project.ipynb`.

## Bước 2: Quy trình xử lý dữ liệu

Trong bước này, chúng ta sẽ chuẩn bị dữ liệu cho việc huấn luyện mô hình Machine Learning. Mục tiêu là làm sạch, biến đổi và cấu trúc lại dữ liệu một cách tối ưu, đồng thời tuân thủ nghiêm ngặt nguyên tắc tránh rò rỉ dữ liệu (Data Leakage) để đảm bảo mô hình có thể khái quát hóa tốt trên dữ liệu mới.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

sys.path.append(str(Path('../src').resolve()))
from preprocess import (
    CATEGORICAL_FEATURES,
    ENGINEERED_FEATURES,
    MODEL_NUMERICAL_FEATURES,
    RAW_NUMERICAL_FEATURES,
    TARGET_COLUMN,
    build_preprocessor,
    generate_schema,
    load_dataset,
    prepare_training_data,
)

df = load_dataset(Path('../data/housing.csv.zip'))
display(df.head())
print('Shape:', df.shape)
print('Target:', TARGET_COLUMN)
print('Numerical raw features:', RAW_NUMERICAL_FEATURES)
print('Categorical features:', CATEGORICAL_FEATURES)
print('Engineered features:', ENGINEERED_FEATURES)

### Bước 2.1: Kiểm tra ban đầu & Loại bỏ cột lỗi

In [ ]:
print('Data types:')
display(df.info())
print('Describe:')
display(df.describe())
print('Missing values:')
display(df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
print(f'Target column: {TARGET_COLUMN}')

**Trả lời câu hỏi:**

**Dữ liệu hiện tại có đáng tin không?**
Dữ liệu hiện tại có vẻ đáng tin cậy về mặt cấu trúc và kiểu dữ liệu. Hầu hết các cột số đã được định dạng đúng. Tuy nhiên, như đã phân tích trong EDA, có một số vấn đề cần xử lý:
*   Cột `total_bedrooms` có giá trị thiếu.
*   Biến mục tiêu `median_house_value` có vẻ bị cắt trần ở mức 500,000 USD.
*   Các biến `median_income` và `housing_median_age` cũng có dấu hiệu bị cắt trần hoặc tập trung ở các giá trị tối đa.

Những vấn đề này cần được xử lý cẩn thận trong các bước tiếp theo của quy trình tiền xử lý để đảm bảo chất lượng của mô hình ML.

**Cột nào bị loại bỏ vì rò rỉ thông tin mục tiêu?**
Trong bộ dữ liệu nhà ở California này, không có cột nào khác ngoài biến mục tiêu `median_house_value` mà rõ ràng là 'rò rỉ thông tin mục tiêu' (tức là một cột được tạo ra hoặc có mặt sẵn trong dữ liệu mà bản chất là một phiên bản trực tiếp của biến mục tiêu và sẽ không có sẵn trong thực tế khi mô hình dự đoán). Do đó, chúng ta sẽ không loại bỏ thêm cột nào ở bước này.

### Bước 2.2: Chia dữ liệu (Train/Test Split)

In [ ]:
prepared = prepare_training_data(df)
print('X_train:', prepared['X_train'].shape)
print('X_test:', prepared['X_test'].shape)
print('y_train:', prepared['y_train'].shape)
print('y_test:', prepared['y_test'].shape)
print('X_train_fe:', prepared['X_train_fe'].shape)
print('X_test_fe:', prepared['X_test_fe'].shape)
print('X_train_processed:', prepared['X_train_processed'].shape)
print('X_test_processed:', prepared['X_test_processed'].shape)

**Trả lời câu hỏi:**

**Vì sao chọn tỉ lệ chia này?**
Tôi đã chọn tỉ lệ chia 80% cho tập huấn luyện (train) và 20% cho tập kiểm tra (test). Tỉ lệ 80/20 là một lựa chọn phổ biến và cân bằng, cho phép mô hình có đủ dữ liệu để học các mẫu (patterns) trong tập huấn luyện (80% dữ liệu) và sau đó được đánh giá trên một tập dữ liệu đủ lớn, chưa từng thấy trước đó (20% dữ liệu) để ước tính hiệu suất khái quát hóa của nó một cách khách quan. Việc sử dụng `random_state=42` đảm bảo rằng việc chia dữ liệu này là lặp lại được (reproducible).

**Có sử dụng stratify không (nếu là bài toán phân loại)?**
Trong trường hợp này, chúng ta đang giải quyết một bài toán hồi quy (dự đoán giá nhà trung bình - một biến liên tục), không phải bài toán phân loại. Do đó, tham số `stratify` không được sử dụng. `stratify` thường được dùng trong các bài toán phân loại để đảm bảo tỉ lệ các lớp trong tập huấn luyện và tập kiểm tra được duy trì tương tự như trong tập dữ liệu gốc, giúp tránh tình trạng mất cân bằng lớp trong các tập con.

### Bước 2.3: Thiết lập Pipeline Tiền xử lý (ColumnTransformer)

In [ ]:
preprocessor = build_preprocessor()
print(preprocessor)
print('Numerical model features:', MODEL_NUMERICAL_FEATURES)

**Trả lời câu hỏi:**

**Vì sao chọn phương pháp điền khuyết này thay vì cách khác?**
*   **`total_bedrooms` (numerical):** Tôi chọn `SimpleImputer` với `strategy='median'` (trung vị). Lý do là `total_bedrooms` là một biến số và như đã thấy trong EDA, các biến số trong dữ liệu nhà ở thường có phân bố lệch hoặc có ngoại lai (ví dụ, một số khu vực có rất ít phòng ngủ so với tổng số phòng). Trung vị ít bị ảnh hưởng bởi các giá trị ngoại lai hơn so với trung bình (`mean`), do đó nó cung cấp một ước lượng đáng tin cậy hơn cho giá trị thiếu. Việc điền bằng một giá trị cố định (`constant`) có thể gây ra thông tin sai lệch và thêm nhiễu vào dữ liệu.
*   **`ocean_proximity` (categorical):** Mặc dù hiện tại cột này không có giá trị thiếu, nếu có, tôi sẽ sử dụng `strategy='most_frequent'` (mode) để điền, vì đây là cách tiêu chuẩn để điền giá trị thiếu cho biến phân loại.

**Vì sao chọn kiểu mã hóa này cho từng cột?**
*   **`ocean_proximity` (categorical):** Tôi chọn `OneHotEncoder`. Lý do là đây là một biến phân loại **định danh** (nominal), không có mối quan hệ thứ bậc rõ ràng giữa các hạng mục (ví dụ: 'NEAR BAY' không lớn hơn hay nhỏ hơn 'INLAND'). `OneHotEncoder` sẽ tạo ra các cột nhị phân mới cho mỗi hạng mục, giúp mô hình ML không giả định bất kỳ mối quan hệ thứ bậc nào mà không tồn tại, đồng thời tránh các vấn đề về khoảng cách giả tạo khi các giá trị số được gán cho các hạng mục.

**Model dự kiến nào cần chuẩn hóa, model nào không cần? Tại sao lại chọn scaler này?**
*   **Các mô hình cần chuẩn hóa:** Hầu hết các mô hình Machine Learning dựa trên khoảng cách (distance-based) hoặc gradient descent đều rất nhạy cảm với thang đo của các đặc trưng. Ví dụ: K-Nearest Neighbors (KNN), Support Vector Machines (SVM), hồi quy tuyến tính (Linear Regression) với Regularization (Lasso, Ridge), Mạng nơ-ron (Neural Networks). Việc chuẩn hóa đảm bảo rằng không có đặc trưng nào chiếm ưu thế quá mức chỉ vì thang đo của nó lớn hơn, giúp tối ưu hóa thuật toán hội tụ và cải thiện hiệu suất.
*   **Các mô hình không cần chuẩn hóa:** Các mô hình dựa trên cây (Tree-based models) như Decision Tree, Random Forest, Gradient Boosting (XGBoost, LightGBM) thường không yêu cầu chuẩn hóa vì chúng hoạt động dựa trên các điểm chia (split points) và mối quan hệ thứ bậc của dữ liệu, không phải khoảng cách Euclid.
*   **Tại sao chọn `StandardScaler`?** Tôi chọn `StandardScaler` vì nó biến đổi dữ liệu sao cho phân bố của mỗi đặc trưng có trung bình bằng 0 và độ lệch chuẩn bằng 1. Điều này hữu ích khi các đặc trưng có phân bố gần với phân bố chuẩn hoặc khi có các giá trị ngoại lai không quá cực đoan. Nó làm cho các đặc trưng có thang đo khác nhau trở nên tương đương, giúp các thuật toán tối ưu hóa hoạt động hiệu quả hơn. `MinMaxScaler` cũng là một lựa chọn, nhưng nó nhạy cảm hơn với các giá trị ngoại lai cực đoan vì nó nén dữ liệu vào một phạm vi cố định (thường là [0, 1]).

### Bước 2.4: Tạo/Chọn đặc trưng (Feature Engineering)

In [ ]:
display(prepared['X_train_fe'].head())
print('Engineered missing values:')
display(prepared['X_train_fe'][ENGINEERED_FEATURES].isnull().sum())

**Trả lời câu hỏi:**

**Đặc trưng mới mang lại thông tin gì?**
Chúng ta đã tạo ra ba đặc trưng mới từ các đặc trưng số hiện có để nắm bắt các mối quan hệ tỷ lệ và mật độ quan trọng trong dữ liệu nhà ở:

1.  **`rooms_per_household` (Số phòng trung bình mỗi hộ gia đình):** Đặc trưng này cung cấp thông tin về không gian sống trung bình cho mỗi hộ gia đình. Một khu vực có nhiều phòng hơn mỗi hộ gia đình có thể gợi ý về những ngôi nhà lớn hơn hoặc nhiều không gian cá nhân hơn, điều này thường liên quan đến giá trị bất động sản cao hơn. Nó giúp chuẩn hóa yếu tố 'số phòng' theo quy mô hộ gia đình, giảm thiểu ảnh hưởng của kích thước tuyệt đối của khu dân cư.

2.  **`bedrooms_per_room` (Tỷ lệ phòng ngủ so với tổng số phòng):** Đặc trưng này cho biết mật độ phòng ngủ trong một ngôi nhà hoặc khu vực. Tỷ lệ thấp có thể cho thấy nhiều không gian chung hơn (ví dụ: phòng khách, phòng ăn lớn), trong khi tỷ lệ cao có thể chỉ ra một ngôi nhà được thiết kế hiệu quả hơn hoặc tập trung vào số lượng phòng ngủ. Đặc trưng này giúp mô hình hiểu được cấu trúc bên trong của ngôi nhà.

3.  **`population_per_household` (Số người trung bình mỗi hộ gia đình):** Đặc trưng này mô tả mật độ dân số trong một hộ gia đình. Các khu vực có số người trên mỗi hộ gia đình cao hơn có thể liên quan đến các loại nhà ở hoặc điều kiện sống khác nhau, có thể ảnh hưởng đến giá trị nhà. Nó giúp phản ánh quy mô và cấu trúc xã hội của các hộ gia đình.

Những đặc trưng mới này giúp giải quyết vấn đề đa cộng tuyến tiềm ẩn đã được phát hiện trong heatmap tương quan của EDA (giữa `total_rooms`, `total_bedrooms`, `population`, `households`). Bằng cách kết hợp các đặc trưng gốc thành các tỷ lệ có ý nghĩa, chúng ta có thể cung cấp cho mô hình những thông tin có giá trị hơn và giảm bớt sự phụ thuộc lẫn nhau của các đặc trưng, từ đó có thể cải thiện khả năng dự đoán của mô hình.

### Bước 2.5: Xử lý mất cân bằng (Nếu có)

In [ ]:
# Bước này dành cho bài toán phân loại để xử lý mất cân bằng lớp.
# Đây là bài toán hồi quy, nên không có khái niệm 'mất cân bằng lớp'.
print("Đây là bài toán hồi quy (dự đoán giá nhà trung bình - một biến liên tục).")
print("Do đó, không có khái niệm 'mất cân bằng lớp' để xử lý ở bước này.")
print("Các kỹ thuật lấy mẫu lại (resampling) như SMOTE hay RandomUnderSampler CHỈ được áp dụng trong bài toán phân loại và CHỈ trên tập huấn luyện để tránh rò rỉ dữ liệu.")

**Trả lời câu hỏi:**

**Tình trạng mất cân bằng ra sao?**
Trong bài toán này, chúng ta đang thực hiện hồi quy, tức là dự đoán một giá trị liên tục (`median_house_value`). Khái niệm 'mất cân bằng lớp' (class imbalance), thường được đề cập trong các bài toán phân loại khi một hoặc nhiều lớp có số lượng mẫu ít hơn đáng kể so với các lớp khác, không áp dụng trực tiếp cho bài toán hồi quy. Thay vào đó, chúng ta quan tâm đến phân bố của biến mục tiêu (`median_house_value`).

Như đã thấy trong EDA, biến mục tiêu của chúng ta có phân bố lệch phải và đặc biệt có rất nhiều giá trị bị cắt trần ở mức 500,000 USD. Đây không phải là vấn đề mất cân bằng lớp mà là một vấn đề về phân bố dữ liệu và giới hạn của việc thu thập dữ liệu. Việc này có thể ảnh hưởng đến khả năng dự đoán của mô hình ở các khoảng giá cao.

**Lưu ý nhắc lại rằng kỹ thuật lấy mẫu lại (resampling) CHỈ được áp dụng trên tập train.**
Đúng vậy, nếu đây là một bài toán phân loại và cần xử lý mất cân bằng lớp, các kỹ thuật lấy mẫu lại (resampling) như Oversampling (ví dụ: SMOTE) hoặc Undersampling (ví dụ: RandomUnderSampler) **phải được áp dụng CHỈ trên tập huấn luyện (X_train, y_train)**. Việc áp dụng các kỹ thuật này trên tập kiểm tra (X_test, y_test) sẽ dẫn đến rò rỉ dữ liệu (data leakage), làm cho việc đánh giá hiệu suất mô hình không còn khách quan, vì mô hình sẽ được đánh giá trên dữ liệu đã được 'thấy' hoặc 'tạo ra' từ tập huấn luyện.

### Bước 2.6: Lưu trữ Đầu ra (Artifacts)

In [ ]:
schema = generate_schema(df)
print('Schema preview:')
print(schema.keys())
print('Required raw features:', schema['required_raw_features'])
print('Model feature names:', schema['model_feature_names'])

**Trả lời câu hỏi:**

**Vì sao cần lưu Pipeline tiền xử lý và Schema dữ liệu?**

1.  **Lưu Pipeline tiền xử lý (`preprocessor.pkl`):**
    *   **Tính nhất quán:** Việc lưu trữ pipeline tiền xử lý đảm bảo rằng cùng một quy trình tiền xử lý (điền giá trị thiếu, tạo đặc trưng mới, mã hóa, chuẩn hóa) đã được áp dụng trên tập huấn luyện sẽ được áp dụng chính xác cho dữ liệu mới (ví dụ: dữ liệu trong môi trường sản phẩm khi mô hình hoạt động) hoặc tập kiểm tra trong tương lai. Điều này ngăn chặn sự không nhất quán giữa dữ liệu huấn luyện và dữ liệu dự đoán, điều có thể dẫn đến hiệu suất mô hình kém.
    *   **Tái sử dụng:** Pipeline đã được fit trên tập huấn luyện, có nghĩa là nó đã học các thông số như trung vị để điền khuyết, min/max hoặc trung bình/độ lệch chuẩn để chuẩn hóa từ tập huấn luyện. Chúng ta chỉ cần `transform` dữ liệu mới, mà không cần `fit` lại, giúp tiết kiệm thời gian và tài nguyên.
    *   **Đơn giản hóa triển khai:** Khi triển khai mô hình vào AI Service, chúng ta chỉ cần tải pipeline này lên cùng với mô hình, thay vì phải viết lại toàn bộ logic tiền xử lý, giúp quá trình triển khai nhanh chóng và ít lỗi hơn.

2.  **Lưu Schema dữ liệu (`schema.json`):**
    *   **Xác thực dữ liệu đầu vào:** File schema này hoạt động như một hợp đồng (contract) cho dữ liệu đầu vào. Nó định nghĩa các đặc trưng dự kiến (tên cột), kiểu dữ liệu của chúng, các khoảng giá trị hợp lệ (min-max cho biến số), và danh sách các nhãn cho biến phân loại. Khi nhận dữ liệu mới trong môi trường sản phẩm, AI Service có thể sử dụng schema này để xác thực rằng dữ liệu đầu vào có định dạng chính xác và nằm trong các giới hạn cho phép trước khi đưa vào mô hình. Điều này giúp phát hiện sớm các lỗi dữ liệu và ngăn chặn mô hình đưa ra dự đoán sai lệch do dữ liệu không hợp lệ.
    *   **Tài liệu hóa:** Schema là một tài liệu rõ ràng về cấu trúc dữ liệu mà mô hình mong đợi, giúp các nhà phát triển hoặc kỹ sư khác hiểu và tương tác với hệ thống một cách hiệu quả.